# 🧪 Teste de Lógica - MercadoExpress

Este notebook serve para validar os componentes internos (Banco de Dados, Excel Engine e Agente LlamaIndex) sem precisar rodar o servidor Flask ou o Docker do Waha.


In [1]:
# 1. Configuração Inicial e Variáveis de Ambiente
import os
from dotenv import load_dotenv
import pandas as pd

# Carrega as chaves do .env
load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    print("⚠️ AVISO: OPENAI_API_KEY não encontrada no .env")
else:
    print("✅ Variáveis de ambiente carregadas.")


✅ Variáveis de ambiente carregadas.


## 2. Teste do Motor Excel (Leitura de Produtos)
Vamos verificar se o sistema consegue ler o `produtos.xlsx` e encontrar itens.


In [2]:
from src.excel_engine import buscar_produto_no_excel, carregar_catalogo

# Teste 1: Carregamento bruto
df = carregar_catalogo()
if df is not None:
    print(f"✅ Catálogo carregado com {len(df)} produtos.")
    display(df.head(3)) # Mostra as 3 primeiras linhas se estiver no Jupyter
else:
    print("❌ Erro ao carregar data/produtos.xlsx")

# Teste 2: Busca Específica
termo = "leite"
resultado = buscar_produto_no_excel(termo)
print(f"\n--- Busca por '{termo}' ---")
print(resultado)


✅ Catálogo carregado com 192 produtos.


,Setor,Produto,Marca,Unidade,Preço
0,Açougue,Carne Moída,Friboi,1Kg,"24,90"
1,Açougue,Filé de Peito de Frango,Seara,1Kg,"14,90"
2,Açougue,Linguiça Toscana,Seara,1Kg,17.99



--- Busca por 'leite' ---
🔎 Encontrei estas opções para 'leite':

- Leite Desnatado (Itambé) - 1L: R$ 4.69
- Leite Integral (QualiMax) - 500ml: R$ 28.45
- Creme de Leite (Selecta) - 250g: R$ 4.24
- Leite Condensado (Marca Boa) - 400ml: R$ 9.86
- Leite Desnatado (QualiMax) - 1L: R$ 7,50



## 3. Teste do Banco de Dados (SQLite)
Vamos inicializar o banco, salvar um pedido fictício e consultar.


In [3]:
from src.database import init_db, salvar_pedido_db, buscar_pedidos_quinzena
import sqlite3

# 1. Inicializa (Cria tabelas se não existirem)
init_db()
print("✅ Banco de dados inicializado.")

# 2. Simula salvar um pedido
id_pedido = salvar_pedido_db(
    nome="Samuel Teste",
    telefone="11999999999",
    endereco="Rua dos Testes, 123",
    itens="1x Leite (R$ 4.69)",
    total=4.69,
    metodo="Pix"
)
print(f"✅ Pedido de teste salvo com ID: {id_pedido}")

# 3. Consulta para ver se gravou
pedidos = buscar_pedidos_quinzena()
print(f"📦 Total de pedidos no banco: {len(pedidos)}")
print("Último pedido:", pedidos[-1])


✅ Banco de dados inicializado.
✅ Pedido de teste salvo com ID: 22
📦 Total de pedidos no banco: 22
Último pedido: (22, 'Samuel Teste', '11999999999', 'Rua dos Testes, 123', '1x Leite (R$ 4.69)', 4.69, 'Pix', '2026-02-28 12:56:16')


## 4. Teste de Geração de Relatórios
Verifica se o arquivo .txt é gerado na pasta `data/relatorios`.


In [4]:
from src.gerar_relatorio import gerar_relatorio_txt
import os

msg = gerar_relatorio_txt()
print(msg)

# Lista arquivos na pasta para confirmar
if os.path.exists("data/relatorios"):
    print("📂 Arquivos na pasta relatorios:", os.listdir("data/relatorios"))


Relatório gerado com sucesso: data/relatorios/relatorio_20260228_0956.txt
📂 Arquivos na pasta relatorios: ['relatorio_20260113_2247.txt', 'relatorio_20260117_1519.txt', 'relatorio_20260128_2027.txt', 'relatorio_20260217_2127.txt', 'relatorio_20260217_2133.txt', 'relatorio_20260218_2019.txt', 'relatorio_20260219_2011.txt', 'relatorio_20260219_2033.txt', 'relatorio_20260228_0956.txt']


## 5. Simulação do Agente (LlamaIndex)
Aqui testamos o cérebro da IA. Vamos simular uma conversa completa para ver se ele segue as regras:
1. Pede nome/telefone no início.
2. Consulta o Excel corretamente.
3. Só pede endereço no final.
4. Salva o pedido.


In [1]:
from src.agent import get_agent

# Inicializa o agente
print("🤖 Inicializando Agente...")
agent = get_agent()

# Função auxiliar para imprimir bonito
def chat_simulado(mensagem_usuario):
    print(f"\n👤 Usuário: {mensagem_usuario}")
    resposta = agent.chat(mensagem_usuario)
    print(f"🤖 Agente: {resposta}")

# --- INÍCIO DA SIMULAÇÃO ---

# 1. Saudação (Agente deve pedir nome/telefone)
chat_simulado("Oi, gostaria de fazer um pedido")

# 2. Fornecendo dados
chat_simulado("Meu nome é Samuel e meu zap é 1198888-7777")

# 3. Pedindo produto (Deve consultar o Excel)
chat_simulado("Vocês tem queijo mussarela?")

# 4. Adicionando ao carrinho
chat_simulado("Vou querer 1kg de mussarela da Tirolez")

# 5. Tentando fechar (Agente deve pedir endereço)
chat_simulado("Pode fechar a conta")

# 6. Fornecendo endereço e pagando
chat_simulado("Moro na Av. Paulista, 1000. Vou pagar no Crédito.")


d:\proj_ia\mercado_express\MERCADO_PROJECT\.venv\lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'validate_default' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'validate_default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(


🤖 Inicializando Agente...


d:\proj_ia\mercado_express\MERCADO_PROJECT\.venv\lib\site-packages\llama_index\agent\openai\base.py:144: DeprecationWarning: Call to deprecated class OpenAIAgent. (OpenAIAgent has been deprecated and is not maintained.

`FunctionAgent` is the recommended replacement.

See the docs for more information on updated agent usage: https://docs.llamaindex.ai/en/stable/understanding/agent/)
  return cls(
d:\proj_ia\mercado_express\MERCADO_PROJECT\.venv\lib\site-packages\deprecated\classic.py:184: DeprecationWarning: Call to deprecated class AgentRunner. (AgentRunner has been deprecated and is not maintained.

This implementation will be removed in a v0.13.0.

See the docs for more information on updated agent usage: https://docs.llamaindex.ai/en/stable/understanding/agent/)
  return old_new1(cls, *args, **kwargs)
d:\proj_ia\mercado_express\MERCADO_PROJECT\.venv\lib\site-packages\llama_index\agent\openai\step.py:213: DeprecationWarning: Call to deprecated class OpenAIAgentWorker. (OpenAIAgentWo


👤 Usuário: Oi, gostaria de fazer um pedido
Added user message to memory: Oi, gostaria de fazer um pedido
🤖 Agente: Olá! Para começar, poderia me informar seu nome e telefone, por favor?

👤 Usuário: Meu nome é Samuel e meu zap é 1198888-7777
Added user message to memory: Meu nome é Samuel e meu zap é 1198888-7777
🤖 Agente: Obrigado, Samuel! Como posso ajudar você hoje? Quais produtos você gostaria de adicionar ao seu carrinho?

👤 Usuário: Vocês tem queijo mussarela?
Added user message to memory: Vocês tem queijo mussarela?
=== Calling Function ===
Calling function: tool_consultar_preco with args: {"produto":"queijo mussarela"}
Got output: 🔎 Encontrei estas opções para 'queijo mussarela':

- Queijo Mussarela (Tirolez) - 1Kg: R$ 39.90


🤖 Agente: Temos o Queijo Mussarela da marca Tirolez, 1Kg, por R$ 39,90. Gostaria de adicionar ao seu carrinho?

👤 Usuário: Vou querer 1kg de mussarela da Tirolez
Added user message to memory: Vou querer 1kg de mussarela da Tirolez
🤖 Agente: Ótimo! Adicion